# 第3章：バラバラに計算して、あとでまとめる（サンプルごとの勾配→平均）

この章では、バネのデータ $(1, 2)$ と $(2, 4)$ を使って、

- **サンプルごとに勾配（修正指示）を計算**し、
- **最後に平均して1回だけ更新**する

という流れを、手計算と同じ形のコードで確認します。

これは将来的に **ミニバッチ学習** や **深層学習** を理解する上での最重要ポイントです。

---

## 1. データの再確認

まずは手元にある2つのデータと、スタート地点の重み $w$ を用意します。

- データ1: $(x_1, y_1) = (1, 2)$
- データ2: $(x_2, y_2) = (2, 4)$
- 重み（学習で更新する値）: $w = 0.0$

In [ ]:
# データ1: (x1, y1) = (1, 2)
x1, y1 = 1.0, 2.0

# データ2: (x2, y2) = (2, 4)
x2, y2 = 2.0, 4.0

# スタートの重み
w = 0.0

print(f"x1, y1 = ({x1}, {y1})")
print(f"x2, y2 = ({x2}, {y2})")
print(f"start w = {w}")

---

## 2. データ1（1個目）の「言い分」を聞く

まず、データ $(1, 2)$ **だけ**を見て、今の $w$ をどう変えてほしいか（勾配）を計算します。

この章では、モデルはとてもシンプルに

$$\hat{y} = w \times x$$

損失（誤差）は二乗誤差で

$$L = (\hat{y} - y)^2$$

とします。

データ1については：

- 予測値: $\hat{y}_1 = w \times 1$
- 誤差の式: $L_1 = (\hat{y}_1 - y_1)^2 = (w \times 1 - 2)^2$
- このデータだけの勾配: 

$$L'_1 = \frac{dL_1}{dw} = 2\,(\hat{y}_1 - y_1)\,x_1$$

> ここで $\hat{y}_1$ は「予測値」なので、$w x_1$ を毎回書かずに $\hat{y}_1$ と置き換えると見通しが良くなります。

In [ ]:
# データ1における勾配の計算
# L1 = (y_pred1 - y1)^2
# y_pred1 = w*x1
# dL1/dw = 2*(y_pred1 - y1)*x1

y_pred1 = w * x1
grad1 = 2 * (y_pred1 - y1) * x1

print(f"データ1の予測値 y_pred1: {y_pred1}")
print(f"データ1からの修正指示（勾配）: {grad1}")

> **解説**
>
> 今は $w=0$ なので、予測は $\hat{y}_1 = 0$。
> 本当は $y_1=2$ なので、かなり足りません。
>
> 勾配が `-4.0` ということは、
>
> - 損失を減らすには $w$ を **増やす方向** に動かすべき
>
> という「修正指示」だと読めます（マイナス勾配なので、更新式 $w \leftarrow w - \text{lr}\times \text{grad}$ では $w$ が増えます）。

---

## 3. データ2（2個目）の「言い分」を聞く

次に、データ $(2, 4)$ **だけ**を見て、同じように計算します。

- 予測値: $\hat{y}_2 = w \times 2$
- 誤差の式: $L_2 = (\hat{y}_2 - y_2)^2 = (w \times 2 - 4)^2$
- このデータだけの勾配:

$$L'_2 = \frac{dL_2}{dw} = 2\,(\hat{y}_2 - y_2)\,x_2$$

> データ1と同様に、$w x_2$ の代わりに「予測値」$\hat{y}_2$ を使って書くと、式の意味が追いやすくなります。

In [ ]:
# データ2における勾配の計算
# L2 = (y_pred2 - y2)^2
# y_pred2 = w*x2
# dL2/dw = 2*(y_pred2 - y2)*x2

y_pred2 = w * x2
grad2 = 2 * (y_pred2 - y2) * x2

print(f"データ2の予測値 y_pred2: {y_pred2}")
print(f"データ2からの修正指示（勾配）: {grad2}")

> **解説**
>
> データ2では $x$ が大きい分、同じ $w$ でも誤差の影響が大きくなりやすいです。
>
> 今は $w=0$ なので $\hat{y}_2=0$、本当は $y_2=4$。
> その結果、勾配は `-16.0` になり、
>
> - 「もっと強く $w$ を増やしてほしい！」
>
> という意見になっています。

---

## 4. みんなの意見を「平均」する

2つのデータから、別々に勾配（修正指示）が出ました。

AIでは普通、**どのデータも公平に扱う**ために、次の順番で平均を作ります。

1. **全員分を足し合わせる（蓄積）**
2. **人数（データ数）で割る（平均）**

式で書くと

$$\text{total} = \text{grad}_1 + \text{grad}_2$$
$$\text{mean} = \frac{\text{total}}{2}$$

となります。

In [ ]:
# 1. まず全員分を足し合わせる（蓄積）
total_grad = grad1 + grad2

# 2. 人数（データ数）で割って平均を出す
mean_grad = total_grad / 2

print(f"みんなの意見をまとめた平均勾配: {mean_grad}")

---

## 5. 実際に一歩進む（更新）

平均した勾配（`mean_grad`）を使って、重み $w$ を1回だけ更新します。

更新式は

$$w \leftarrow w - \mathrm{lr}\times \text{mean}$$

です（`lr` は learning rate / 学習率）。

In [ ]:
lr = 0.1
w = w - lr * mean_grad

print(f"更新後の w: {w}")

> **解説**
>
> ここでは `w` が `1.0` になったはずです。
>
> 重要なのは、
>
> - 「全データの損失を1個の巨大な式にしてから微分」
>
> しても、
>
> - 「サンプルごとに勾配を出して最後に平均」
>
> しても、**同じ更新結果**になることです。
>
> この性質があるので、実装では後者（サンプルごと→平均）がとても扱いやすくなります。

---

## 6. なぜこの「バラバラ作戦」が最強なのか？

一見、手間が増えたように見えますが、実はこれが超重要です。

| 方法 | 特徴 | 弱点 |
| :--- | :--- | :--- |
| 一括計算（全部まとめてから微分） | 全データを1つの数式にする | データが増えると式が巨大化して扱いにくい |
| サンプルごとの平均（1つずつ→最後に平均） | 1つ1つの計算が単純で、足し合わせれば良い | 並列化もしやすく、大規模データに強い |

### まとめ（この章のゴール）

1. 各データごとに「今の $w$ についての不満（勾配）」を計算する。
2. 全員の不満を合計して、人数で割る（平均）。
3. その平均不満を解消する方向に、$w$ を動かす。

---

## 次のステップ

この「1つずつ計算して、あとでまとめる」作業、データが増えると手で書くのが大変ですよね。

次回は **PyTorchの `Tensor`** を使って、同じことを一気に（そして自動微分で）できる形に進化させます。

---

# （続き）第3章：勾配を「溜める」という仕組み

承知いたしました。`for` 文という「自動化」の魔法を使う前に、まずはその中身を **手動で1ステップずつ** 展開してみましょう。

やっていることは、

- リストの **0番目** と **1番目** を順番に処理して
- 出てきた勾配を **1つの変数に足し合わせていく**

という、極めてシンプルな作業です。

## ねらい

AIの学習では、複数のデータから出た「修正してほしい！」という意見（勾配）を、一つの場所に **溜めて（蓄積して）** から、最後にその平均を使って重みを更新します。

この「溜めていく」感覚を、まずは手動で体験してみましょう。

---

## 1. 準備（リストでデータを持つ）

さっきまでは `(x1, y1)`, `(x2, y2)` と名前を分けていました。

ここからは、実務でよくやるように

- 入力を `x = [...]`
- 正解を `y = [...]`

の **リスト** で持ちます。

さらに、

- 学習率 `lr`
- データ数 `N`

も用意しておきます（平均を出すときに使います）。

In [ ]:
x = [1.0, 2.0]
y = [2.0, 4.0]

# 重みはここでリセット（この続き部分だけで完結するように）
w = 0.0

lr = 0.1
N = len(x)

print(f"x = {x}")
print(f"y = {y}")
print(f"N = {N}")
print(f"start w = {w}")

---

## 2. 勾配の「ゴミ箱」をリセット（空にする）

まず、今回の学習で使う勾配を溜める変数 `total_grad` を用意し、`0.0` で空っぽにします。

この **「最初に0にする」** のが超重要です。

- 前回の学習で溜まった勾配が残っていると、今回の計算に混ざってしまう

からです。

（将来 PyTorch を使うときは、ここが `optimizer.zero_grad()` に相当します。）

In [ ]:
# 1. 勾配を溜める変数をリセット（ゴミ箱を空にする）
total_grad = 0.0

print(f"start total_grad = {total_grad}")

---

## 3. 【1人目】`i = 0` のデータを処理（手動）

リストの 0 番目のデータを取り出して、

1. 予測 `y_pred`
2. このデータ専用の勾配 `grad`
3. 勾配を `total_grad` に足し算（蓄積）

を行います。

ここでのポイントは、最後の

- `total_grad = total_grad + grad`

という **上書き加算** です。

「箱（total_grad）に、票（grad）を入れる」イメージで見てください。

In [ ]:
# i = 0 のとき（1人目）
i = 0
xi = x[i]
yi = y[i]

y_pred = w * xi
grad = 2 * (y_pred - yi) * xi

# ★ 勾配を溜める（現在の total_grad に加算）
total_grad = total_grad + grad

print(f"i={i}")
print(f"xi={xi}, yi={yi}")
print(f"y_pred={y_pred}")
print(f"データ{i}の勾配 grad: {grad}")
print(f"現在の total_grad: {total_grad}")

---

## 4. 【2人目】`i = 1` のデータを処理（手動）

次に 1 番目のデータを処理します。

ここでも同じ手順で `grad` を作りますが、重要なのは **`total_grad` を0に戻さない** ことです。

- すでに入っている分（1人目の票）に、2人目の票を **さらに足す**

というイメージです。

In [ ]:
# i = 1 のとき（2人目）
i = 1
xi = x[i]
yi = y[i]

y_pred = w * xi
grad = 2 * (y_pred - yi) * xi

# ★ 勾配をさらに溜める（これまでの total_grad に追加）
total_grad = total_grad + grad

print(f"i={i}")
print(f"xi={xi}, yi={yi}")
print(f"y_pred={y_pred}")
print(f"データ{i}の勾配 grad: {grad}")
print(f"合計された total_grad: {total_grad}")

> **解説：蓄積のイメージ**
>
> `total_grad` は、いわばアンケートの集計箱です。
>
> - `i=0` さんが「-4.0」という票を入れる
> - `i=1` さんが「-16.0」という票を入れる
>
> その結果、箱の中身は合計で **-20.0** になります。

---

## 5. 平均を出して、重みを更新する

全員分の意見（勾配）が `total_grad` に溜まったので、最後に

1. 平均勾配 `mean_grad` を作る（`total_grad / N`）
2. その平均で $w$ を更新する

を行います。

（将来 PyTorch を使うときは、更新が `optimizer.step()` に相当します。）

In [ ]:
# 1) 全データの平均勾配を出す
mean_grad = total_grad / N

# 2) 重みを更新する
w = w - lr * mean_grad

print(f"mean_grad: {mean_grad}")
print(f"1回学習後の w: {w:.4f}")

---

## 6. まとめ：これを自動化するのが `for` 文！

今は `i = 0` と `i = 1` で、ほぼ同じコードを 2 回書きました。

データが 100 個あったら、この手動作業を 100 回書くことになり、とても大変です。

そこで、全く同じ動きをする **「自動化バージョン（for ループ）」** に書き換えます。

ポイントは同じで、順番はこうです：

1. `total_grad` を 0.0 にリセット
2. ループで `grad` を計算して `total_grad` に足し続ける（蓄積）
3. ループが終わったら平均して更新する

下のコードは、さっき手でやったことと **意味が完全に同じ** です。

In [ ]:
x = [1.0, 2.0]
y = [2.0, 4.0]

# もう一度、初期状態から（for版が単体で動くようにリセット）
w = 0.0

total_grad = 0.0

for i in range(N):
    xi = x[i]
    yi = y[i]

    y_pred = w * xi
    grad = 2 * (y_pred - yi) * xi

    # 算術演算子 += を使ってスマートに加算
    total_grad = total_grad + grad

mean_grad = total_grad / N
w = w - lr * mean_grad

print(f"for版 mean_grad: {mean_grad}")
print(f"for版 更新後の w: {w:.4f}")

In [ ]:
def model_function(w, x):
    return w * x

In [ ]:
x = [1.0, 2.0]
y = [2.0, 4.0]

# もう一度、初期状態から（for版が単体で動くようにリセット）
w = 0.0

total_grad = 0.0

for i in range(N):
    xi = x[i]
    yi = y[i]

    y_pred = model_function(w, xi)
    grad = 2 * (y_pred - yi) * xi

    # 算術演算子 += を使ってスマートに加算
    total_grad = total_grad + grad

mean_grad = total_grad / N
w = w - lr * mean_grad

print(f"for版 mean_grad: {mean_grad}")
print(f"for版 更新後の w: {w:.4f}")

並列で計算する

In [ ]:
import numpy as np

# 1. データの準備（NumPyでまとめて処理）
x = np.array([1.0, 2.0])
y = np.array([2.0, 4.0])

def model_linear(w, x):
    return w * x

w = 0.0

def criterion_mse(y_pred, y):
    loss = np.mean((y_pred - y) ** 2)
    return loss

# --- 学習の実行 ---

# 手順A: 予測して、正解とのズレを測る
y_pred = model_linear(w, x)

loss = criterion_mse(y_pred, y)

# 手順B: 勾配を計算する
all_grads = 2 * (y_pred - y) * x
mean_grad = np.mean(all_grads)

# 手順C: 実際に重みを動かす
lr = 0.1
w = w - lr * mean_grad

# 結果確認
print(f"NumPy版 誤差: {loss}")
print(f"NumPy版 更新後の w: {w:.4f}")

---

## 7. ここまでの最重要ワンポイント

この章で覚える形は、これです：

**「0でリセット → ループで足す（蓄積） → 平均 → 更新」**

この形がそのまま、

- ミニバッチ学習（小さなまとまりごとに平均を取る）
- 深層学習（大量のパラメータの勾配を同じように溜める）

につながっていきます。

次はこの流れを、PyTorch の Tensor と自動微分で“手計算なし”にしていきましょう。